# Supplementary Figure 7A: all-section spatial replication

This notebook rebuilds the spot-level source data from the fixed Tangram outputs and draws `FigureS7A_all_section_spatial_replication_legend_corrected.pdf`. It does not read the finished revision figure source table or call an external plotting script.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

PROJECT_ROOT = Path("/mnt/disk18t/lr_xcy/riku/kirc_205")
FIGS7_DIR = PROJECT_ROOT / "main_codes/figs7"
SOURCE_DIR = FIGS7_DIR / "source_data"
FIGURE_DIR = FIGS7_DIR / "figures"
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TANGRAM_TABLES = Path("/mnt/disk18t/lr_xcy/riku/codex_research/CM_analysis_2/spatial/cm_assignment_analysis_method16_tangram_pseudobulk_subtype_mapping_score_top100_all_cells/tables")
METHOD = "method16_tangram_pseudobulk_subtype_mapping_score_top100_all_cells_use_X"
SECTIONS = ["h46t", "j38t", "r114t", "r29t", "r51t", "s15t", "x49t", "x98t", "y12t", "y27t", "y7t", "z43t"]
CM1_COL = "CMact__joint_01_sharedCM"
JUN_COL = "EPIfrac__Epi_JUN"
VIM_COL = "EPIfrac__Epi_VIM"

def percentile_rank(values):
    values = np.asarray(values, dtype=float)
    return stats.rankdata(values, method="average") / len(values)

def build_figures7a_source_data():
    tables = {}
    positive_epithelial_values = []
    for sample in SECTIONS:
        path = TANGRAM_TABLES / f"{sample}_{METHOD}_spot_scores.csv"
        if not path.exists():
            raise FileNotFoundError(path)
        table = pd.read_csv(path, index_col=0)
        table.index = table.index.astype(str)
        required = {"array_row", "array_col", CM1_COL, JUN_COL, VIM_COL}
        missing = sorted(required - set(table.columns))
        if missing:
            raise KeyError(f"{sample}: missing columns {missing}")
        epithelial = table[[column for column in table.columns if column.startswith("EPIfrac__")]].to_numpy(float)
        positive_epithelial_values.append(epithelial[epithelial > 0])
        tables[sample] = table

    epsilon = 0.5 * min(float(values.min()) for values in positive_epithelial_values)
    parts = []
    summary = []
    for sample in SECTIONS:
        table = tables[sample]
        cm1 = table[CM1_COL].to_numpy(float)
        jun = table[JUN_COL].to_numpy(float)
        vim = table[VIM_COL].to_numpy(float)
        balance = np.log((jun + epsilon) / (vim + epsilon))
        cm1_percentile = percentile_rank(cm1)
        balance_percentile = percentile_rank(balance)
        rho, p_value = stats.spearmanr(cm1_percentile, balance_percentile)
        summary.append({"sample": sample, "n_spots": len(table), "spearman_rho": rho, "p_value": p_value})
        parts.append(pd.DataFrame({
            "sample": sample,
            "spot_id": table.index,
            "array_row": table["array_row"].to_numpy(int),
            "array_col": table["array_col"].to_numpy(int),
            "CM1_activity": cm1,
            "Epi_JUN_fraction": jun,
            "Epi_VIM_fraction": vim,
            "JUN_VIM_logratio": balance,
            "CM1_percentile": cm1_percentile,
            "JUN_VIM_logratio_percentile": balance_percentile,
            "epsilon": epsilon,
        }))

    spots = pd.concat(parts, ignore_index=True)
    section_summary = pd.DataFrame(summary)
    if spots["sample"].nunique() != 12:
        raise RuntimeError("Expected 12 tumour sections")
    spots.to_csv(SOURCE_DIR / "FigureS7A_spot_source_data.csv.gz", index=False, compression="gzip")
    section_summary.to_csv(SOURCE_DIR / "FigureS7A_section_correlation_summary.csv", index=False)
    return spots, section_summary, epsilon

figures7a_spots, figures7a_section_summary, figures7a_epsilon = build_figures7a_source_data()
print(f"spots={len(figures7a_spots):,}; sections={figures7a_spots['sample'].nunique()}; positive correlations={(figures7a_section_summary['spearman_rho'] > 0).sum()}/12; epsilon={figures7a_epsilon:.6g}")


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

COLORS = {"cm1": "#3D6D9C", "jun": "#E67E22", "dark": "#263238", "mid": "#66717B"}
OVERLAP = "#7B4E8C"
CATEGORY_COLORS = ["#E8ECEF", COLORS["cm1"], COLORS["jun"], OVERLAP]

def configure_style():
    regular = Path("/home/lr_xcy/.local/share/fonts/codex_trajectory_20260813/Arial.TTF")
    bold = regular.with_name("Arialbd.TTF")
    for path in (regular, bold):
        if path.exists():
            font_manager.fontManager.addfont(path)
    mpl.rcParams.update({
        "font.family": "Arial" if regular.exists() else "DejaVu Sans",
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9.5,
        "xtick.labelsize": 8.5,
        "ytick.labelsize": 8.5,
        "legend.fontsize": 8.5,
        "axes.linewidth": 0.75,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "figure.facecolor": "white",
        "savefig.facecolor": "white",
    })

def header(fig, letter, title, subtitle):
    fig.text(0.035, 0.965, letter, fontsize=17, fontweight="bold", color=COLORS["dark"], va="top")
    fig.text(0.10, 0.965, title, fontsize=14, fontweight="bold", color=COLORS["dark"], va="top")
    fig.text(0.10, 0.915, subtitle, fontsize=9.5, color=COLORS["mid"], va="top")

def coordinates(frame):
    return frame["array_col"].to_numpy(float), -frame["array_row"].to_numpy(float) * np.sqrt(3.0)

def bivariate_categories(frame):
    cm1_high = frame["CM1_percentile"].to_numpy() >= 0.5
    jun_leaning = frame["JUN_VIM_logratio_percentile"].to_numpy() >= 0.5
    return cm1_high.astype(int) + 2 * jun_leaning.astype(int)

def spatial_axis(axis, frame, values):
    x, y = coordinates(frame)
    axis.scatter(x, y, c=values, cmap=ListedColormap(CATEGORY_COLORS), vmin=-0.5, vmax=3.5, s=3.2, linewidth=0, rasterized=True)
    axis.set_aspect("equal")
    axis.set_xlim(x.min() - 3, x.max() + 3)
    axis.set_ylim(y.min() - 3, y.max() + 3)
    axis.axis("off")

configure_style()
fig, axes = plt.subplots(3, 4, figsize=(10.0, 7.6))
fig.subplots_adjust(left=0.03, right=0.98, bottom=0.12, top=0.83, wspace=0.06, hspace=0.18)
header(fig, "A", "The aligned co-niche recurs across all 12 tumour sections", "Purple spots are simultaneously CM1-high and JUN-leaning within their section")
for axis, section in zip(axes.flat, SECTIONS):
    section_data = figures7a_spots.loc[figures7a_spots["sample"].eq(section)]
    spatial_axis(axis, section_data, bivariate_categories(section_data))
    axis.set_title(section, fontsize=9.5, fontweight="bold", color=COLORS["dark"], pad=1)
legend = [
    Patch(facecolor=CATEGORY_COLORS[0], label="CM1-low + VIM-leaning"),
    Patch(facecolor=CATEGORY_COLORS[1], label="CM1 high only"),
    Patch(facecolor=CATEGORY_COLORS[2], label="JUN-leaning only"),
    Patch(facecolor=CATEGORY_COLORS[3], label="aligned co-niche"),
]
fig.legend(handles=legend, ncol=4, loc="lower center", frameon=False, bbox_to_anchor=(0.5, 0.035))
positive_sections = int((figures7a_section_summary["spearman_rho"] > 0).sum())
fig.text(0.98, 0.04, f"{positive_sections}/12 section correlations > 0", ha="right", color=OVERLAP, fontweight="bold", fontsize=10)
output_stem = FIGURE_DIR / "FigureS7A_all_section_spatial_replication_legend_corrected"
fig.savefig(output_stem.with_suffix(".pdf"), bbox_inches="tight")
fig.savefig(output_stem.with_suffix(".svg"), bbox_inches="tight")
fig.savefig(output_stem.with_suffix(".png"), dpi=450, bbox_inches="tight")
plt.close(fig)
print(output_stem.with_suffix(".pdf"))
